# Built-in Tools (CrewAI)

This notebook accompanies [Agent Foundry](https://agent-foundry-pi.vercel.app).

In [ ]:
!pip install -q crewai crewai-tools

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## Tools on agents and tasks

`FileReadTool` reads local files and needs no third-party search API. Assign it on the **agent** with `tools=[...]`, and optionally on a **task** with `tools=[...]` to override the agent’s tools for that task only.

The sample file path points at this repo’s `README.md` (two levels up from `notebooks/crewai/`).

In [ ]:
import os

from crewai import Agent, Crew, Process, Task
from crewai_tools import FileReadTool

root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sample_path = os.path.join(root, "README.md")

file_tool = FileReadTool()

reader = Agent(
    role="Documentation Reader",
    goal="Read local files and summarize them accurately",
    backstory="You never invent file contents; you only report what you read.",
    tools=[file_tool],
    verbose=True,
)

task_read = Task(
    description=(
        f"Use the file read tool to read this path and give a one-sentence summary: {sample_path}"
    ),
    expected_output="One sentence describing what the README is about.",
    agent=reader,
    tools=[file_tool],
)

task_no_tool = Task(
    description="In one sentence, explain that the previous task already summarized the README — do not call any tools.",
    expected_output="One sentence, no tool usage.",
    agent=reader,
    tools=[],
)

crew = Crew(
    agents=[reader],
    tasks=[task_read, task_no_tool],
    process=Process.sequential,
    verbose=True,
)

result = crew.kickoff()
print(result)

## Key takeaways

- **Agent `tools`**: default capabilities the model can use across tasks.
- **Task `tools`**: overrides the agent’s list for that task (`[]` means no tools).
- **`FileReadTool`**: good for local demos without extra API keys beyond your LLM.
- Pair other `crewai_tools` (e.g. `SerperDevTool`) with the right env vars when you move to web search.